<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

Code based on https://github.com/pytorch/examples/blob/master/mnist/main.py

In this exercise, we are using high-level abstractions from torch.nn like nn.Linear.
Note: during the next lab session we will go one level deeper and implement more things
with bare hands.

Tasks:

 1. Read the code.

 2. Check that the given implementation reaches 95% test accuracy for architecture input-128-128-10 after few epochs.

 3. Add the option to use SGD with momentum instead of ADAM.

 4. Experiment with different learning rates. Use the provided TrainingVisualizer
 to plot the learning curves and gradient-to-weight ratios. Compare visualizations
 for different learning rates for both ADAM and SGD with momentum.

 5. Parameterize the constructor by a list of sizes of hidden layers of the MLP.
 Note that this requires creating a list of layers as an attribute of the Net class,
 and one can't use a standard Python list containing nn.Modules (why?).
 Check torch.nn.ModuleList.

If you run this notebook locally then you may need to install some packages.
It may be achieved by adding the following code cell to the notebook and running it:
```
!pip install torch torchvision plotly ipywidgets
```

This notebook will work faster on a Colab GPU (or TPU).
However, start with tests just on CPU, and try to end your GPU sessions as soon as you stop needing them (otherwise you may use all your free GPU time for this week).

In [25]:
import math
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import plotly.graph_objects as go
from torch import Tensor
from torchvision import datasets, transforms

# Allow custom (non-ipywidget) widgets.
if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()

# For reproducibility.
torch.manual_seed(1)

## Tensors
In ML, *tensor* is just a short name for an n-dimensional array.

In other areas like physics, it might mean a multilinear operation that can be represented, after a choice of basis, by an ndarray; this is where we would see all the extra stuff (like covariance/contravariance), like for example on the Wikipedia article for "tensor". For us, it's usually just a bunch of numbers.

torch.Tensor is can be thought of as a wrapper around numpy ndarray, but:
* the interface is slightly different (like `np.expand_dims(x, dim=n-1)` vs `torch.unsqueeze(x, axis=-1)`);
* the data can be stored in GPU memory (or others);
* it can be part of a computation graph: a computed Tensor has pointers to other Tensors it was computed from, and to functions computing partial derivatives.

All the operations defined for a torch.Tensor (`@`, `.arctan`, `.unsqueeze`, and a hundred others) can also run on a GPU and other backends.

In [26]:
x_np = np.array([[0, 1, 2], [3, 4, 5]])
x = torch.tensor(
    [[0, 1, 2], [3, 4, 5]]
)  # Also: torch.rand, ones, zeros, rand_like, ...
display(x_np, x)

array([[0, 1, 2],
       [3, 4, 5]])

tensor([[0, 1, 2],
        [3, 4, 5]])

In [27]:
torch.from_numpy(x_np).equal(x), np.array_equal(x_np, x.numpy())

(True, True)

In [28]:
x.shape, x.dtype, x.device, x.requires_grad

(torch.Size([2, 3]), torch.int64, device(type='cpu'), False)

Most operations have an in-place variant with the same name, but with an underscore.
This forgets the original value, making the computation of gradients impossible, so most of the time we don't use in-place operations.

In [29]:
x.add_(2)
display(x)

tensor([[2, 3, 4],
        [5, 6, 7]])

## Autograd
Operations on torch tensors remember the computation graph.
This allows to compute the gradient of e.g. a loss node over all other nodes with loss.backward().

In [30]:
x = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], requires_grad=True)
y = torch.sigmoid(x)
loss = ((y - 1) ** 2).mean()
print("x", x.grad_fn, ", y", y.grad_fn, ", loss", loss.grad_fn)
print(loss.grad_fn.next_functions)  # type: ignore
print(loss.grad_fn.next_functions[0][0].next_functions)  # type: ignore

x None , y <SigmoidBackward0 object at 0x0000028D43E37970> , loss <MeanBackward0 object at 0x0000028D45DD3730>
((<PowBackward0 object at 0x0000028D43E379D0>, 0),)
((<SubBackward0 object at 0x0000028D43E37A00>, 0),)


The method Tensor.backward():
* for each node with `requires_grad`: computes the gradients from each .grad_fn and propagates it using the chain rule.
* for each leaf node (`requires_grad` with no predecessors that `requires_grad`): accumulates the gradient into the tensor’s `.grad` attribute.



In [31]:
loss.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

We say *accumulates* not *stores*, because if .grad already exists, `backward()` adds to it, instead of replacing it.

In [32]:
loss2 = (x * 0).mean()
loss2.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

During (for example) evaluation, you don't need to store the computational graph, so you can disable it:

In [33]:
with torch.no_grad():
    y = x * x

print(y.grad_fn)

None


In some situations you may need more fine-grained control.
You can detach a tensor from the graph, to treat it like a constant tensor.

In [34]:
y = x * x
print(y.grad_fn, y.requires_grad)
y = y.detach()  # Not in-place, returns a new tensor with the same data.
print(y.grad_fn, y.requires_grad)

<MulBackward0 object at 0x0000028D43F108E0> True
None False


## Modules
A torch **module** (not *model*) is essentially just an object with a `forward()` method, usually `forward(x: Tensor) -> Tensor`.
A network (model) is built as a large module that contains and calls smaller modules, which themselves contain even smaller modules.

Basically any callable could be a module, but subclassing `nn.Module` adds features like:
* automatically registering and enumerating submodules and trainable parameters,
* serializing and storing parameters into files,
* adding hooks in the middle of existing models.

In [35]:
class MyLinear(nn.Module):
    """
    This is essentially the same as nn.Linear.

    Compare: https://github.com/pytorch/pytorch/blob/v2.9.0/torch/nn/modules/linear.py#L53
    """

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty((out_features, in_features)))
        self.bias = nn.Parameter(torch.empty(out_features))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # The initialization used by PyTorch by default, including non-zero biases.
        bound = 1 / math.sqrt(self.in_features)
        nn.init.uniform_(self.weight, -bound, bound)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, input: Tensor) -> Tensor:
        """
        Input shape: (B*, in_features).
        Output shape: (B*, out_features).
        """
        return input @ self.weight.T + self.bias


We define the module behaviour in the **`forward()`** method, not in the special `__call__()`.<br>
However, we then use the module directly as a callable, `module()`.<br>
This will call `module.forward()`, but can also calls hooks, if any are defined.

In [36]:
module = MyLinear(2, 3)
x = torch.rand((7, 5, 2))
y = module(x)
y.shape

torch.Size([7, 5, 3])

The class **nn.Parameter** just wraps a tensor to indicate that this is a trainable parameter, and changes the default `requires_grad`.<br>
When an nn.Parameter is assigned to a field of an nn.Module, it automatically registers it,
so that all parameters of this modules and its submodules can be easily accessed.

In [37]:
for i, (name, param) in enumerate(module.named_parameters()):
    print(f'{i}. "{name}":\t{param}\n')

0. "weight":	Parameter containing:
tensor([[ 0.3643, -0.3121],
        [-0.1371,  0.3319],
        [-0.6657,  0.4241]], requires_grad=True)

1. "bias":	Parameter containing:
tensor([-0.1455,  0.3597,  0.0983], requires_grad=True)



## A basic Net

In [38]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, 28, 28).
        Output: class probabilities (after softmax), shape (B, 10).
        """
        x = torch.flatten(x, start_dim=1)  # (B, 728)
        x = self.fc1(x)  # (B, 128)
        x = F.relu(x)
        x = self.fc2(x)  # (B, 128)
        x = F.relu(x)
        x = self.fc3(x)  # (B, 10)
        output = F.log_softmax(x, dim=1)
        return output

Assigning to a nn.Module to a field of nn.Module automatically registers it as a submodule.<br>
You can iterate over all submodules (recursively) with `.named_modules()` or over direct children only with `.named_children()`.

In [39]:
net = Net()
for i, (name, submodule) in enumerate(net.named_modules()):
    print(f'{i}. "{name}":\t{submodule}')

0. "":	Net(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)
1. "fc1":	Linear(in_features=784, out_features=128, bias=True)
2. "fc2":	Linear(in_features=128, out_features=128, bias=True)
3. "fc3":	Linear(in_features=128, out_features=10, bias=True)


The `.train()` and `.eval()` methods set the `training` flag of a module and all its submodules.<br>
This changes the behaviour of certain modules like BatchNorm and Dropout (see next labs).

Forgetting to switch the training mode is a common bug that can cause weird behaviour.

In [40]:
net.train()
print(net.fc1.training)
net.eval()
print(net.fc1.training)

True
False


Note that operations on a module (`.train()`, `.eval()`, `.to(dtype)`, `.to(device)`) modify it in-place.<br>
Operations on a tensor without a trailing underscore (including `.to(dtype)`, `.to(device)`) return a new tensor and leave the original unchanged


## The generalized Net()

In [41]:
class Net(nn.Module):
    def __init__(self, sizes: list[int] | None = [128, 128]):
        super().__init__()
        self.layers = nn.ModuleList()
        if sizes:
            self.layers.append(nn.Linear(in_features=784, out_features=sizes[0]))
            for i in range(0, len(sizes)-1):
                self.layers.append(nn.Linear(in_features=sizes[i], out_features=sizes[i+1]))
            self.layers.append(nn.Linear(in_features=sizes[-1], out_features=10))
        else:
            self.layers.append(nn.Linear(in_features=784, out_features=10))

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, 28, 28).
        Output: class probabilities (after softmax), shape (B, 10).
        """
        x = torch.flatten(x, start_dim=1)  # (B, 728)
        for i in range(len(self.layers)-1):
            x = F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (B, 10)
        output = F.log_softmax(x, dim=1)
        return output

## Train and test code

In [42]:
# @title <small>Helper class: TrainingVisualizer</small>


class TrainingVisualizer:
    def __init__(self, log_interval: int = 10):
        self.log_interval = log_interval
        self.train_loss_fig = self.init_line_plot(
            title="Training loss", xaxis_title="Step"
        )
        self.grad_to_weight_fig = self.init_line_plot(
            title="Gradient standard deviation to weight standard deviation ratio at 1st layer",
            xaxis_title="Step",
            yaxis_title="Gradient to weight ratio (log scale)",
            yaxis_type="log",
        )
        self.test_acc_fig = self.init_line_plot(
            title="Test accuracy", x=[], xaxis_title="Epoch", mode="lines+markers"
        )

        # Parameters related to current tracked model and its training
        self.first_linear_layer = None
        self.lr = None
        self.trace_idx = -1

    def init_line_plot(
        self,
        title: str,
        x=None,
        xaxis_title: str | None = None,
        yaxis_title: str | None = None,
        yaxis_type: str = "linear",
        mode: str = "lines",
    ) -> go.FigureWidget:
        fig = go.Figure()
        fig.update_layout(
            title=title,
            title_x=0.5,
            xaxis_title=xaxis_title,
            yaxis_title=yaxis_title,
            height=400,
            width=1500,
            margin=dict(b=10, t=60),
        )
        fig.update_yaxes(type=yaxis_type)
        # We cannot add new traces dynamically because Colab has a problem with widgets
        # from plotly (traces added dynamically are rendered twice).
        # As an ugly workaround we create a lot of empty traces and update them later
        # with actual data. Empty traces are not plotted.
        for _ in range(25):
            fig.add_trace(go.Scatter(x=x, y=[], showlegend=True, mode=mode))

        fig_widget = go.FigureWidget(fig)
        display(fig_widget)
        return fig_widget

    def track_model(
        self, model: torch.nn.Module, optimizer: torch.optim.Optimizer, lr: float
    ) -> None:
        """
        Start tracking training metrics for a new model.
        """

        for field in model.__dict__["_modules"].values():
            if isinstance(field, nn.Linear):
                self.first_linear_layer = field
                break
            elif isinstance(field, nn.ModuleList):
                self.first_linear_layer = field[0]
                break

        self.lr = lr
        self.trace_idx += 1

        optim_name = type(optimizer).__name__
        self.train_loss_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.grad_to_weight_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.test_acc_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"

    def plot_gradients_and_loss(self, batch_idx: int, loss: float) -> None:
        if batch_idx % self.log_interval == 0:
            self.train_loss_fig.data[self.trace_idx].y += (loss,)

            layer = self.first_linear_layer
            grad_to_weight_ratio = (
                self.lr * layer.weight.grad.std() / layer.weight.std()
            ).item()

            self.grad_to_weight_fig.data[self.trace_idx].y += (grad_to_weight_ratio,)

    def plot_accuracy(self, epoch: int, accuracy: float) -> None:
        self.test_acc_fig.data[self.trace_idx].x += (epoch,)
        self.test_acc_fig.data[self.trace_idx].y += (accuracy,)

In [43]:
def train_epoch(
    model: torch.nn.Module,
    device: torch.device,
    train_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    log_interval: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to the device (e.g. GPU #0) where model parameters are.
        data, target = data.to(device), target.to(device)

        # Since `backward()` will accumuate gradients, remember to reset them.
        optimizer.zero_grad()

        # Forward pass.
        output = model(data)  # Logs of class probabilites, shape (B, 10)
        loss = F.nll_loss(output, target)  # Shape (1,)

        # Backward pass.
        loss.backward()

        visualizer.plot_gradients_and_loss(batch_idx, loss.item())

        # For each parameter, updates the tensor's value using its .grad.
        optimizer.step()

        if batch_idx % log_interval == 0:
            if verbose:
                done, total = batch_idx * len(data), len(train_loader.dataset)
                print(
                    f"Train Epoch: {epoch} [{done}/{total} images ({done / total:.0%})]\t"
                    + f"Loss: {loss.item():.6f}"
                )

In [44]:
def test(
    model: torch.nn.Module,
    device: torch.device,
    test_loader: torch.utils.data.DataLoader,
    epoch: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.eval()
    test_loss = 0
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            # Accumulate sum of dataitem losses and only divide by n_total at the end.
            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            n_correct += (pred == target).sum().item()
            n_total += len(target)

    test_loss /= n_total

    if verbose:
        print(
            f"\nTest loss: {test_loss:.4f}, Accuracy: {n_correct}/{n_total} ({n_correct / n_total:.0%})\n"
        )
    visualizer.plot_accuracy(epoch, 100.0 * n_correct / n_total)

## Hyperparameters

In [45]:
# Training uses more memory than test (due to gradient computations),
# so we can set test_batch_size to be larger.
epochs = 5
batch_size = 256
test_batch_size = 1000
lr = 1e-2
log_interval = 10

# Check for CUDA / TPU (Google) / MPS (Apple) / XPU (Intel) / ... accelerator.
# (TPU as of 2026 gives spurious warnings about lss.item() accessing .grad - ignore them).
device = (
    torch.accelerator.current_accelerator(check_available=True)  #
    or torch.device("cpu")
)
use_accel = device != torch.device("cpu")
print(use_accel, device)

False cpu


## Dataset and dataloader
Any sequence of dataitems (and even some iterables) can be used as a ***dataset***.<br>
A dataitem can be for example an (input image, output label) pair.<br>
Torch datasets usually allow passing a transformation: it is then applied whenever you retrieve an item with `dataset[i]`.

A ***dataloader*** manages parallel loading (it can create worker threads that call `dataset[i]` in parallel), batching, shuffling, sometimes moving data between devices.

In [46]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=(0.1307,), std=(0.3081,))]
)
train_dataset = datasets.MNIST(
    "../data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST("../data", train=False, transform=transform)
len(train_dataset), len(test_dataset)

(60000, 10000)

In [47]:
kwargs = {"num_workers": 1, "pin_memory": True} if use_accel else {}
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, **kwargs
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=test_batch_size, **kwargs
)

## Running it all

In [48]:
model = Net().to(device)

optimizer = optim.SGD(model.parameters(), lr=lr)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '86854bc0-cf77-4db5-b6aa-459488562b56',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4b3da147-cd53-4a9e-80e9-e70fdbadc936',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2e2060e7-4453-4be8-8693-e36bd59c534a',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '04787f7e-99ce-4138-b060-d29c1a65c786',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '98e03e0f-24e9-43d2-9935-64eb7ceefe62',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8c0ac55b-e47d-49cd-8446-818abf7314e9',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2939c93e-0f07-4388-a5f2-82798a8ce087',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c7ebfa6d-e952-4296-8026-59556d4e45c2',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'bcc6906e-c11d-4fce-aee4-b7041f000982',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b9ccc9fb-be22-44f0-a306-60e9fd81afc5',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '3cd75308-8902-44e8-b678-8b562aea4fe9',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6bd41b22-da37-44ef-a3ac-e014c20e4dd5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'cd65dc4e-7bdd-4688-ab64-f48a12e35236',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '95887c3b-fe46-4d04-9c8a-bbf3cdce1a81',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.314002
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 2.292223
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 2.264758
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 2.245541
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 2.223753
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 2.200149
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 2.172465
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 2.123429
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 2.075716
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 2.018897
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 1.956960
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 1.922170
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 1.839516
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 1.774920
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 1.703322
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 1.591161
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 1.539864
Train Epoch: 1 [43520/60000 images (73%)]

### The same with SGD optimizer

In [49]:
model = Net().to(device)

optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c3227bf8-a19e-463b-8742-3fc8db12bff7',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'eaf060a1-4c4d-4694-91ef-9775967cd918',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'a8fe26db-a1d3-41c4-9980-df5b865a8d09',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '9ab39da9-2c31-492b-b1a3-8b9a917f7af9',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '0262e2a7-e840-46bb-8882-d6ca689204f6',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '83d6a662-91a4-4c47-b193-8a1d5dc8ac92',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'fdbbf338-9fc8-4682-a992-fa330a87f88e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '73a6bb30-3449-4f46-a291-39d98ec68a9d',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e0c40a61-ff7f-4370-8d4a-9b891eb9abf0',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'ccee5dc1-fbd4-41ed-82f4-6063de05ab1f',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'f098efb0-2624-48e9-88cc-16ff4553590e',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '133d9309-a22c-4315-a9b1-13e03e457706',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4a3d1462-9d84-47d1-b1db-7d69e42892e0',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '08b35a8c-bd7c-4171-8179-591ad37d3d03',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.298934
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 2.186904
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 1.911841
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 1.280300
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.782149
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.660748
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.483191
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.411053
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.457285
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.412225
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.379998
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.348693
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.344527
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.310522
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.378266
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.308914
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.349268
Train Epoch: 1 [43520/60000 images (73%)]

## Dodatkowe porównania: optymalizator, learning rate i architektura

Każdy nowy model jest trenowany przez **3 epoki** (albo inną liczbę z `COMPARISON_EPOCHS`) na pełnym zbiorze treningowym.

Funkcja pomocnicza korzysta z istniejącego treningu i wizualizacji. Każde uruchomienie tworzy nowy model i optymalizator. Stałe ziarno i osobny generator - stała kolejność minibatchy dla porównywanych konfiguracji.


In [50]:
import pandas as pd

COMPARISON_EPOCHS = 3
COMPARISON_SEED = 1


def run_comparison(hidden_sizes, optimizer_name, learning_rate, visualizer):
    torch.manual_seed(COMPARISON_SEED)
    comparison_model = Net(sizes=list(hidden_sizes)).to(device)

    if optimizer_name == "Adam":
        comparison_optimizer = optim.Adam(
            comparison_model.parameters(), lr=learning_rate
        )
    elif optimizer_name == "SGD":
        comparison_optimizer = optim.SGD(
            comparison_model.parameters(), lr=learning_rate, momentum=0.9
        )
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    # A fresh generator keeps minibatch order independent of model initialization.
    comparison_loader = torch.utils.data.DataLoader(
        train_loader.dataset, batch_size=train_loader.batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(COMPARISON_SEED), num_workers=0,
    )

    visualizer.track_model(comparison_model, comparison_optimizer, learning_rate)
    hidden_label = "-".join(map(str, hidden_sizes)) or "no hidden layers"
    optimizer_label = "SGD (momentum=0.9)" if optimizer_name == "SGD" else "Adam"
    label = f"{optimizer_label}, lr={learning_rate:g}, hidden={hidden_label}"
    for figure in (
        visualizer.train_loss_fig,
        visualizer.grad_to_weight_fig,
        visualizer.test_acc_fig,
    ):
        figure.data[visualizer.trace_idx].name = label

    for comparison_epoch in range(1, COMPARISON_EPOCHS + 1):
        train_epoch(
            comparison_model, device, comparison_loader, comparison_optimizer,
            comparison_epoch, log_interval, visualizer, verbose=False,
        )
        test(
            comparison_model, device, test_loader, comparison_epoch,
            visualizer, verbose=False,
        )
        accuracy = float(visualizer.test_acc_fig.data[visualizer.trace_idx].y[-1])
        print(f"{label} | epoch {comparison_epoch}/{COMPARISON_EPOCHS}: {accuracy:.2f}%")

    return {
        "optimizer": optimizer_label,
        "hidden_layers": str(list(hidden_sizes)),
        "learning_rate": learning_rate,
        "parameters": sum(p.numel() for p in comparison_model.parameters()),
        "final_accuracy_%": accuracy,
    }


### 1. Learning rate: Adam i SGD z momentum

Dla architektury `784 → 128 → 128 → 10` porównujemy Adam z learning rate `0.0001`, `0.001`, `0.01` oraz SGD z momentum `0.9` i learning rate `0.001`, `0.01`, `0.1`.

In [ ]:
lr_comparison_visualizer = TrainingVisualizer(log_interval=log_interval)
lr_comparison_results = []

for comparison_optimizer_name, comparison_rates in (
    ("Adam", (1e-4, 1e-3, 1e-2)),
    ("SGD", (1e-3, 1e-2, 1e-1)),
):
    for comparison_lr in comparison_rates:
        lr_comparison_results.append(
            run_comparison(
                [128, 128], comparison_optimizer_name, comparison_lr,
                lr_comparison_visualizer,
            )
        )

display(
    pd.DataFrame(lr_comparison_results)
    .sort_values("final_accuracy_%", ascending=False)
    .reset_index(drop=True)
)


FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '212c1847-1db5-427f-9809-c35291243027',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c3de8abc-6ae9-4f34-aa38-ce8ff6f13c6a',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'eb2421e9-693b-4b13-99ee-74994c51bb15',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '537bec07-8011-4aa9-9d28-c1e9ba2dfa16',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '415c1dab-c572-42f7-8059-5f2f57d1c4aa',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '9f0ca59a-761a-45f9-84c1-1c6783f4b97d',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '38e94341-c1b3-40b2-b992-c76aebb9f5c9',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e4074737-bacf-4147-8262-716e897b2a4e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2ed2c2bb-ac7d-41e5-a858-d09fa4acdc38',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6e16ec98-dba1-41d0-9713-2252f50fe9c0',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '53e8e7c3-d929-419c-acdc-9ac6bd403dd8',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7e31e2e1-b0db-4e4a-a4db-360729d2925d',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5ae1725e-01f8-46e4-be3c-aebaccc6f5df',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2620640b-5878-4378-8266-149d7691af5b',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c

Adam, lr=0.0001, hidden=128-128 | epoch 1/3: 88.92%
Adam, lr=0.0001, hidden=128-128 | epoch 2/3: 91.25%
Adam, lr=0.0001, hidden=128-128 | epoch 3/3: 92.19%
Adam, lr=0.001, hidden=128-128 | epoch 1/3: 94.50%
Adam, lr=0.001, hidden=128-128 | epoch 2/3: 96.15%
Adam, lr=0.001, hidden=128-128 | epoch 3/3: 96.66%


### 2. Liczba i szerokość warstw ukrytych

Przy stałym optymalizatorze Adam i learning rate `0.001` porównujemy listy warstw ukrytych: `[]`, `[64]`, `[128]`, `[128, 128]` oraz `[256, 128, 64]`. Pusta lista oznacza bezpośrednie połączenie `784 - 10`. Pozostałe listy badają szerokość i głębokość sieci.

In [ ]:
architecture_comparison_visualizer = TrainingVisualizer(log_interval=log_interval)
architecture_comparison_results = []

for comparison_hidden_sizes in ([], [64], [128], [128, 128], [256, 128, 64]):
    architecture_comparison_results.append(
        run_comparison(
            comparison_hidden_sizes, "Adam", 1e-3,
            architecture_comparison_visualizer,
        )
    )

display(
    pd.DataFrame(architecture_comparison_results)
    .sort_values("final_accuracy_%", ascending=False)
    .reset_index(drop=True)
)


Plotly comparison figure

Plotly comparison figure

Plotly comparison figure

Adam, lr=0.001, hidden=no hidden layers | epoch 1/3: 91.05%
Adam, lr=0.001, hidden=no hidden layers | epoch 2/3: 91.84%
Adam, lr=0.001, hidden=no hidden layers | epoch 3/3: 92.14%
Adam, lr=0.001, hidden=64 | epoch 1/3: 93.42%
Adam, lr=0.001, hidden=64 | epoch 2/3: 95.25%
Adam, lr=0.001, hidden=64 | epoch 3/3: 96.18%
Adam, lr=0.001, hidden=128 | epoch 1/3: 94.02%
Adam, lr=0.001, hidden=128 | epoch 2/3: 95.83%
Adam, lr=0.001, hidden=128 | epoch 3/3: 96.66%
Adam, lr=0.001, hidden=128-128 | epoch 1/3: 94.50%
Adam, lr=0.001, hidden=128-128 | epoch 2/3: 96.15%
Adam, lr=0.001, hidden=128-128 | epoch 3/3: 96.66%
Adam, lr=0.001, hidden=256-128-64 | epoch 1/3: 94.70%
Adam, lr=0.001, hidden=256-128-64 | epoch 2/3: 96.65%
Adam, lr=0.001, hidden=256-128-64 | epoch 3/3: 97.26%


optimizer,hidden_layers,learning_rate,parameters,final_accuracy_%
Adam,"[256, 128, 64]",0.001,242762,97.26
Adam,"[128, 128]",0.001,118282,96.66
Adam,[128],0.001,101770,96.66
Adam,[64],0.001,50890,96.18
Adam,[],0.001,7850,92.14
